# 08 - Corpus y trazabilidad

Este notebook sirve para documentar los repositorios o muestras utilizadas en el TFG.

La idea es tener una tabla sencilla que después puedas llevar a la memoria.

In [1]:
from pathlib import Path
import json
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LEVEL_ORDER = {"A1": 1, "A2": 2, "B1": 3, "B2": 4, "C1": 5, "C2": 6}
LEVEL_ORDER_INV = {v: k for k, v in LEVEL_ORDER.items()}
RADON_RANK_ORDER = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6}
RADON_RANK_ORDER_INV = {v: k for k, v in RADON_RANK_ORDER.items()}

def clean_file_name(path):
    """Devuelve un nombre de fichero comparable entre herramientas."""
    return Path(str(path)).name

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def detect_json_kind(path):
    """Intenta detectar si un JSON parece de Radon o de PyCEFR."""
    try:
        data = load_json(path)
    except Exception:
        return "unknown"
    # Buscar un registro de ejemplo dentro del árbol project/file/[records]
    for project, files in data.items():
        if not isinstance(files, dict):
            continue
        for file_path, records in files.items():
            if isinstance(records, list) and records:
                rec = records[0]
                if isinstance(rec, dict):
                    if {"Class", "Start Line", "End Line", "Level"}.issubset(set(rec.keys())):
                        return "pycefr"
                    if {"type", "rank", "complexity", "lineno", "endline"}.issubset(set(rec.keys())):
                        return "radon"
    return "unknown"

In [2]:
corpus = pd.DataFrame([
    {
        "case": "Analisis-CC-PyCEFR",
        "origen": "Repositorio propio / herramienta del proyecto",
        "tipo": "Código de la herramienta",
        "nivel_esperado": "B1-B2",
        "motivo": "Permite analizar el propio código desarrollado y comprobar cómo se comportan las métricas sobre la herramienta.",
    },
    {
        "case": "4geeks_beginner",
        "origen": "4GeeksAcademy/python-beginner-programming-exercises",
        "tipo": "Ejercicios básicos",
        "nivel_esperado": "A1-A2",
        "motivo": "Contiene ejercicios introductorios útiles para observar constructos simples.",
    },
    {
        "case": "30_days_python",
        "origen": "Asabeneh/30-Days-Of-Python",
        "tipo": "Curso progresivo",
        "nivel_esperado": "A1-B2",
        "motivo": "Está organizado como material de aprendizaje progresivo.",
    },
    {
        "case": "python_projects",
        "origen": "python-online/python-projects",
        "tipo": "Proyectos pequeños",
        "nivel_esperado": "A2-C1",
        "motivo": "Incluye proyectos de distintos niveles, útiles para comparar casos prácticos.",
    },
    {
        "case": "python_algorithms",
        "origen": "david-legend/python-algorithms u otro repositorio de algoritmos",
        "tipo": "Algoritmos y estructuras de datos",
        "nivel_esperado": "B1-C2",
        "motivo": "Aporta funciones con mayor densidad algorítmica y estructuras de control más complejas.",
    },
])
corpus

,case,origen,tipo,nivel_esperado,motivo
0,Analisis-CC-PyCEFR,Repositorio propio / herramienta del proyecto,Código de la herramienta,B1-B2,Permite analizar el propio código desarrollado...
1,4geeks_beginner,4GeeksAcademy/python-beginner-programming-exer...,Ejercicios básicos,A1-A2,Contiene ejercicios introductorios útiles para...
2,30_days_python,Asabeneh/30-Days-Of-Python,Curso progresivo,A1-B2,Está organizado como material de aprendizaje p...
3,python_projects,python-online/python-projects,Proyectos pequeños,A2-C1,"Incluye proyectos de distintos niveles, útiles..."
4,python_algorithms,david-legend/python-algorithms u otro reposito...,Algoritmos y estructuras de datos,B1-C2,Aporta funciones con mayor densidad algorítmic...


In [4]:
corpus.to_csv(OUTPUT_DIR / "08_corpus_trazabilidad.csv", index=False)

## Texto base para la memoria

Para construir el corpus experimental se seleccionaron cinco casos con distinta orientación. Se incluyó código propio de la herramienta, ejercicios introductorios, material progresivo de aprendizaje, proyectos pequeños y ejemplos de algoritmos o estructuras de datos. Esta selección busca cubrir distintos niveles esperados de dificultad sin hacer que el estudio sea inmanejable por tamaño.